In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.ingestion_file_log"

In [0]:
bronze_before = spark.table(BRONZE_TABLE).count()
silver_before = spark.table(SILVER_CORE_TABLE).count()

completed_files_before = (
    spark.table(LOG_TABLE)
    .filter(F.col("file_status") == "completed")
    .count()
)

print("Bronze before:", bronze_before)
print("Silver before:", silver_before)
print("Completed files before:", completed_files_before)

In [0]:
SILVER_TABLES = {
    "core": f"{CATALOG}.{SCHEMA}.silver_events_core",
    "push": f"{CATALOG}.{SCHEMA}.silver_push_events",
    "pull_request": f"{CATALOG}.{SCHEMA}.silver_pull_request_events",
    "issue": f"{CATALOG}.{SCHEMA}.silver_issue_events",
    "issue_comment": f"{CATALOG}.{SCHEMA}.silver_issue_comment_events",
    "watch": f"{CATALOG}.{SCHEMA}.silver_watch_events",
    "fork": f"{CATALOG}.{SCHEMA}.silver_fork_events",
    "release": f"{CATALOG}.{SCHEMA}.silver_release_events"
}

counts_before = {}

for name, table_name in SILVER_TABLES.items():
    counts_before[name] = spark.table(table_name).count()

for name, count in counts_before.items():
    print(name, count)

In [0]:
SILVER_TABLES = {
    "core": f"{CATALOG}.{SCHEMA}.silver_events_core",
    "push": f"{CATALOG}.{SCHEMA}.silver_push_events",
    "pull_request": f"{CATALOG}.{SCHEMA}.silver_pull_request_events",
    "issue": f"{CATALOG}.{SCHEMA}.silver_issue_events",
    "issue_comment": f"{CATALOG}.{SCHEMA}.silver_issue_comment_events",
    "watch": f"{CATALOG}.{SCHEMA}.silver_watch_events",
    "fork": f"{CATALOG}.{SCHEMA}.silver_fork_events",
    "release": f"{CATALOG}.{SCHEMA}.silver_release_events"
}

counts_before = {}

for name, table_name in SILVER_TABLES.items():
    counts_before[name] = spark.table(table_name).count()

for name, count in counts_before.items():
    print(name, count)

In [0]:
CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

SILVER_TABLES = {
    "core": f"{CATALOG}.{SCHEMA}.silver_events_core",
    "push": f"{CATALOG}.{SCHEMA}.silver_push_events",
    "pull_request": f"{CATALOG}.{SCHEMA}.silver_pull_request_events",
    "issue": f"{CATALOG}.{SCHEMA}.silver_issue_events",
    "issue_comment": f"{CATALOG}.{SCHEMA}.silver_issue_comment_events",
    "watch": f"{CATALOG}.{SCHEMA}.silver_watch_events",
    "fork": f"{CATALOG}.{SCHEMA}.silver_fork_events",
    "release": f"{CATALOG}.{SCHEMA}.silver_release_events"
}

counts_before = {
    "core": 613617,
    "push": 591926,
    "pull_request": 4694,
    "issue": 1473,
    "issue_comment": 1586,
    "watch": 1115,
    "fork": 220,
    "release": 371
}

In [0]:
counts_after = {}

for name, table_name in SILVER_TABLES.items():
    counts_after[name] = spark.table(table_name).count()

for name in SILVER_TABLES:
    print(
        name,
        "before =", counts_before[name],
        "after =", counts_after[name],
        "added =", counts_after[name] - counts_before[name]
    )

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"

In [0]:
duplicate_ids_df = (
    spark.table(BRONZE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_ids_df.show(truncate=False)

In [0]:
duplicate_records_df = (
    spark.table(BRONZE_TABLE)
    .join(
        duplicate_ids_df.select("event_id"),
        on="event_id",
        how="inner"
    )
    .select(
        "event_id",
        "event_type",
        "created_at",
        "source_file",
        "source_hour",
        "raw_json"
    )
    .orderBy("event_id", "source_hour")
)

duplicate_records_df.show(
    truncate=False
)

In [0]:
DUP_EVENT_ID = "14196830486"

TEMP_TABLE = f"{CATALOG}.{SCHEMA}.tmp_duplicate_event"

spark.sql(f"""
CREATE OR REPLACE TABLE {TEMP_TABLE}
USING DELTA
AS
SELECT *
FROM {BRONZE_TABLE}
WHERE event_id = '{DUP_EVENT_ID}'
LIMIT 1
""")

spark.sql(f"""
DELETE FROM {BRONZE_TABLE}
WHERE event_id = '{DUP_EVENT_ID}'
""")

spark.sql(f"""
INSERT INTO {BRONZE_TABLE}
SELECT *
FROM {TEMP_TABLE}
""")

spark.sql(f"DROP TABLE {TEMP_TABLE}")

In [0]:
print(
    "Bronze duplicate IDs:",
    spark.table(BRONZE_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Bronze rows:",
    spark.table(BRONZE_TABLE).count()
)

In [0]:
DUP_EVENT_ID = "14196830486"

spark.sql(f"""
DELETE FROM {CATALOG}.{SCHEMA}.silver_events_core
WHERE event_id = '{DUP_EVENT_ID}'
""")

In [0]:
duplicate_core_row_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_id") == DUP_EVENT_ID)
    .select(
        "event_id",
        "event_type",
        F.to_timestamp("created_at").alias("event_ts"),
        F.get_json_object("actor_json", "$.id").cast("long").alias("actor_id"),
        F.get_json_object("actor_json", "$.login").alias("actor_login"),
        F.get_json_object("repo_json", "$.id").cast("long").alias("repo_id"),
        F.get_json_object("repo_json", "$.name").alias("repo_name"),
        F.get_json_object("org_json", "$.id").cast("long").alias("org_id"),
        "public",
        "source_file",
        "source_hour",
        "ingested_at"
    )
)

duplicate_core_row_df.write.mode("append").saveAsTable(
    f"{CATALOG}.{SCHEMA}.silver_events_core"
)

In [0]:
print(
    "Silver Core duplicate IDs:",
    spark.table(f"{CATALOG}.{SCHEMA}.silver_events_core")
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Silver Core rows:",
    spark.table(f"{CATALOG}.{SCHEMA}.silver_events_core").count()
)

In [0]:
SILVER_WATCH_TABLE = f"{CATALOG}.{SCHEMA}.silver_watch_events"

spark.sql(f"""
DELETE FROM {SILVER_WATCH_TABLE}
WHERE event_id = '{DUP_EVENT_ID}'
""")

In [0]:
watch_row_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_id") == DUP_EVENT_ID)
    .select(
        "event_id",
        F.to_timestamp("created_at").alias("event_ts"),
        F.get_json_object("actor_json", "$.id").cast("long").alias("actor_id"),
        F.get_json_object("repo_json", "$.id").cast("long").alias("repo_id"),
        F.get_json_object("payload_json", "$.action").alias("action"),
        "source_file",
        "source_hour",
        "ingested_at"
    )
)

watch_row_df.write.mode("append").saveAsTable(
    SILVER_WATCH_TABLE
)

In [0]:
print(
    "Silver Watch duplicate IDs:",
    spark.table(SILVER_WATCH_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Silver Watch rows:",
    spark.table(SILVER_WATCH_TABLE).count()
)

In [0]:
test_df = spark.createDataFrame(
    [
        ("1", "WatchEvent"),
        ("1", "WatchEvent"),
        ("2", "PushEvent")
    ],
    ["event_id", "event_type"]
)

print("Before dedup:", test_df.count())

deduped_test_df = test_df.dropDuplicates(["event_id"])

print("After dedup:", deduped_test_df.count())

deduped_test_df.orderBy("event_id").show()

In [0]:
from pyspark.sql import functions as F

synthetic_hours_df = spark.createDataFrame(
    [
        ("2026-09-01 00:00:00",),
        ("2026-09-01 01:00:00",),
        ("2026-09-01 02:00:00",),
        ("2026-09-01 04:00:00",),
        ("2026-09-01 05:00:00",),
    ],
    ["source_hour"]
).withColumn(
    "source_hour",
    F.to_timestamp("source_hour")
)

bounds = synthetic_hours_df.agg(
    F.min("source_hour").alias("min_hour"),
    F.max("source_hour").alias("max_hour")
).first()

expected_hours_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(bounds["min_hour"]),
                F.lit(bounds["max_hour"]),
                F.expr("INTERVAL 1 HOUR")
            )
        ).alias("source_hour")
    )
)

missing_hours_test_df = (
    expected_hours_df
    .join(
        synthetic_hours_df,
        on="source_hour",
        how="left_anti"
    )
)

missing_hours_test_df.show(truncate=False)

print(
    "Detected missing hours:",
    missing_hours_test_df.count()
)

In [0]:
late_arrival_df = spark.createDataFrame(
    [
        ("2026-09-01 03:00:00",)
    ],
    ["source_hour"]
).withColumn(
    "source_hour",
    F.to_timestamp("source_hour")
)

synthetic_hours_with_late_df = (
    synthetic_hours_df
    .unionByName(late_arrival_df)
)

bounds_after_late = synthetic_hours_with_late_df.agg(
    F.min("source_hour").alias("min_hour"),
    F.max("source_hour").alias("max_hour")
).first()

expected_hours_after_late_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(bounds_after_late["min_hour"]),
                F.lit(bounds_after_late["max_hour"]),
                F.expr("INTERVAL 1 HOUR")
            )
        ).alias("source_hour")
    )
)

missing_hours_after_late_df = (
    expected_hours_after_late_df
    .join(
        synthetic_hours_with_late_df,
        on="source_hour",
        how="left_anti"
    )
)

missing_hours_after_late_df.show(truncate=False)

print(
    "Missing hours after late arrival:",
    missing_hours_after_late_df.count()
)

In [0]:
retry_test_df = spark.createDataFrame(
    [
        (
            "dbfs:/Volumes/workspace/default/gharchive_raw/synthetic_failed.json.gz",
            "failed"
        )
    ],
    ["source_file", "file_status"]
)

retry_test_df = (
    retry_test_df
    .withColumn(
        "should_retry",
        F.col("file_status") != "completed"
    )
)

retry_test_df.show(truncate=False)

In [0]:
completed_test_df = spark.createDataFrame(
    [
        (
            "dbfs:/Volumes/workspace/default/gharchive_raw/synthetic_completed.json.gz",
            "completed"
        )
    ],
    ["source_file", "file_status"]
)

completed_test_df = (
    completed_test_df
    .withColumn(
        "should_retry",
        F.col("file_status") != "completed"
    )
)

completed_test_df.show(truncate=False)